In [2]:
# Numpy für bessere Berechnungen
import numpy as np
from numpy import exp, sqrt, log, pi

# Weiteres für bessere Rechnungen
from decimal import Decimal, ROUND_CEILING, ROUND_HALF_UP, getcontext # Besonders für sig. Runden
import math

# Berechnungen und Plotting
import scipy.constants as cc

# Zum Auslesen von Dateien und ähnlichem
from pathlib import Path

from pytexit import py2tex
import re


# Besseres Funktionen handling
import sympy as sp

# Display und Output
from IPython.display import display, Math, Latex, HTML

import textwrap

g=9.80984 
Delta_g=0.00002

In [3]:
currentversuch = "255"
base = Path.cwd()                   
Messwerte_path = Path(base.parent.parent.parent/"Messwerte"/currentversuch)      # initializing paths
Ausgabe_path = Path(base/"Diagramme")
print(Messwerte_path)
print(Ausgabe_path)

c:\Users\samue\Documents\PAP\Messwerte\255
c:\Users\samue\Documents\PAP\tex_files\Auswertungen_1\Code\Diagramme


# Delete UTF8 False encoding
Wenn man in Altprotokollen Copy Pasten will, die meistens kein utf8 encoding benutzen, sind ä,ö,ü fucked up. Um das rückgängig zu machen, gibt es in RemoveVocals eine Funktion für.

Runden signifikanter Stellen

In [4]:
def round_to_sigs(val,errVal=None,einheiten=None):
    """
    Funktion zur Rundung eines Fehlers und die Anpassung des Messwertes daran. Diese Funktion wurde etwas umständlicher 
    geschrieben, da python mit Floats und Runden schnell in Probleme rennt. Daher musste hier mit dezimal gearbeitet werden.
    Zudem sollten besonders kleine und große Messwerte in der Dezimalschreibweise geschrieben werden, damit diese auch 
    für Protokolle geiegnet sind.

    Parameter
    ----------
    **val** : float
        Messwert

    **errVal** : float
        Ungenauigkeit des Messwertes

    Return
    ------
    **value_rounded** : str
        Gerundeter Messwert

    **error_rounded** : str
        Gerundete Ungenauigkeit des Messwertes

    **res** : str
        "Messwert \\pm Fehler"
    """
    einheiten_str = einheiten if einheiten is not None else ""
    if errVal is not None:
        # Daten zu Dezimal wechseln, da Floats probleme machen
        val = Decimal(str(val))
        errVal = Decimal(str(errVal))

        exp = int(math.floor(math.log10(float(errVal))))            # Die erste signifikante Stellenposition wird anhand des errVal bestimmt

        if round(float(errVal / (Decimal(10) ** exp))) < 3:         # wenn 1,2,3 erste signifikante Stelle, dann kommt eine zweite Nachkommastellenposition hinzu
            exp -= 1

        scale = Decimal(10) ** exp                                  

        val_round = (val / scale).quantize(Decimal('1'), rounding=ROUND_HALF_UP) * scale            # mit scale wird das Komma so verschoben, dass alle signifikanten Stellen vor dem Komma stehen, und dann alle Nachkommastellen weggerundet werden
        err_round = (errVal / scale).quantize(Decimal('1'), rounding=ROUND_CEILING) * scale

        num = f"\\num{{{val_round}({err_round})}}"
        qty = f"\\qty{{{val_round}({err_round})}}{{{einheiten_str}}}" if einheiten else num

        return float(val_round), float(err_round), num, qty
    else:
        val = Decimal(str(val))
        exp = int(math.floor(math.log10(float(val))))
        if round(float(val / (Decimal(10) ** exp))) < 3:         # wenn 1,2,3 erste signifikante Stelle, dann kommt eine zweite Nachkommastellenposition hinzu
            exp -= 1
        scale = Decimal(10) ** exp   
        val_round = (val / scale).quantize(Decimal('1'), rounding=ROUND_HALF_UP) * scale
        num = f"\\num{{{val_round}}}"
        qty = f"\\qty{{{val_round}}}{{{einheiten_str}}}"if einheiten else num
    return float(val_round), None, num, qty

v_round = np.vectorize(round_to_sigs, otypes=[float, float, object, object], excluded=['einheiten'])

    # wissenschaftliche Notation erstmal vernachlässigen (da häufig Einheiten gewollt sind, die 10^/e Prefixe beinhalten)

Gausssche Fehlerfortpflanzung

In [5]:
def gff(function, errPronePar,latex=True):
    """
    Kann die Fehlerformel einer gegebenen Gleichung bestimmen.

    Parameters
    ----------
    **func** : sympy function
        Funktion dessen Fehler bestimmt werden soll.

    **errPronePar** : Array
        Liste (Array) aller fehlerbehafteten Größen der Gleichung con sp.Symbols
        Diese Werte werden als x_sym, y_sym, z_sym etc. bezeichnet und sind ungleich den Werten für x, y, z.
        Für die Werte wird daher die Bezeichnung x_val, y_val, z_val etc. genutzt und für deren Fehler err_x, err_y, err_z etc.

    Return
    ----------
    **absolut_err** : sympy function
        Gibt die Fehlergleichung des absoluten Fehlers wieder. 

    **relativ_err** : sympy function
        Gibt die Fehlergleichung des relativen Fehlers wieder. 
        
    **errProneParamters** : array
        Liste aller Fehlerbehafteten Größen
    """ 

    error = 0
    errProneParamaters = []

    function = sp.sympify(function)
    variables = errPronePar.split(",")
    variables = sp.symbols(variables)

    for variable in variables:
        Delta = sp.Symbol(f"Delta_{variable}")
        partial = sp.diff(function, variable)                   # Die Funktion wird nach der fehlerbehafteten Variable abgeleitet
        term=sp.UnevaluatedExpr(partial*Delta)**2
        error = error + ((term))                                # Fehler werden quadratisch aufsummiert
        errProneParamaters.append((variable,Delta))
    
    absolute_err=sp.simplify(sp.sqrt(error),rational = True)             
    relative_err=sp.simplify(sp.sqrt(error/function**2),rational = True)
    
    if latex:
        #Prints out the Latex Code for the Functions
        print("Funktion:")
        display(Math(sp.latex(function,long_frac_ratio=2))) 
        print(sp.latex(function))
        print("Absoluter Fehler:")
        display(Math(sp.latex(absolute_err,long_frac_ratio=2)))
        print(sp.latex(absolute_err))
        print("Relativer Fehler:")
        display(Math(sp.latex(relative_err,long_frac_ratio=2)))
        print(sp.latex(relative_err))
    return function,absolute_err, relative_err, errProneParamaters


Sigma-Abweichungen

In [6]:
def sigma_abweichung(p1, err_p1,p2,err_p2):
    """
    Funktion zum berechnen der Sigma-Abweichugn von zwei Messwerten, oder einem Messwert und einem Literaturwert.
    """
    if err_p1 == 0 and err_p2 == 0:
        raise ValueError("Für die Sigma-Abweichung muss mindestens ein Wert fehlerbehaftet sein!")
    else:
        abweichung=Decimal(str(abs(p1 - p2)/(np.sqrt(err_p1 ** 2+err_p2 ** 2))))

    if abweichung == 0:
        return 0.0, "\\num{0}"
    
    exp = int(math.floor(math.log10(float(abweichung))))
    if round(float(abweichung / (Decimal(10) ** exp))) < 3:         # wenn 1,2,3 erste signifikante Stelle, dann kommt eine zweite Nachkommastellenposition hinzu
        exp -= 1
    scale = Decimal(10) ** exp   
    abweichung_round = (abweichung / scale).quantize(Decimal('1'), rounding=ROUND_CEILING) * scale
    res = f"\\num{{{abweichung_round}}}"

    return float(abweichung_round),res



In [7]:

# #Größenvergleich in latex
# def size_comp_str(name1,name2,val1,val2):
#     if val1<val2:
#         return name1+"<"+name2
#     elif val1>val2:
#         return name1+">"+name2
#     else:
#         return name1+"="+name2
   
#Erstellung von Vergleichstabellen in Latex
def compare_table_array(namM_list,nam1, nam2, einheiten, val1_array, err1_array, val2_array, err2_array):
    # 1. Tabellenkopf (Hier werden die Strings einmalig eingesetzt)
    output_header = textwrap.dedent(f"""
        \\begin{{table}}[htbp]
        \\centering
        \\caption{{Vergleich von \\({nam1}\\) und \\({nam2}\\)}}
        \\begin{{tabularx}}{{\\textwidth}}{{ZZZZZS[table-format=1.2]}}
            \\toprule
                 \\text{{Messreihe}} & {nam1}\\;[\\unit{{{einheiten}}}] & {nam2}\\;[\\unit{{{einheiten}}}] & \\text{{abs. Abw.}}\\;[\\unit{{{einheiten}}}] & \\text{{proz. Abw.}}\\;[\\unit{{\\percent}}] & {{S\\(\\;[\\sigma]\\)}} \\\\\\midrule
    """).strip() # strip macht whitespaces am anfang und ende des strings weg
    
    table_rows = []

    # 2. Der Body (Schleife läuft über die Werte-Arrays)
    for namM, val1, err1, val2, err2 in zip(namM_list, val1_array, err1_array, val2_array, err2_array):
        
        VAL1=round_to_sigs(val1,err1,r'')[2]
        if err2!=0:
            VAL2=round_to_sigs(val2,err2,r'')[2]
        else:
            VAL2=val2

        abs=np.abs(val1-val2)
        abs_delta=np.sqrt(err1**2+err2**2)
        if abs_delta!=0:
            SIGMA=sigma_abweichung(val1,err1,val2,err2)[0]
        else:
            SIGMA=0.0

        rel=abs/np.abs(val1)*100
        if abs != 0:
            rel_delta = rel * np.sqrt((abs_delta / abs)**2 + (err1 / val1)**2)
            ABS = round_to_sigs(abs, abs_delta, r'')[2]
            REL = round_to_sigs(rel, rel_delta, r'')[2]
        else:
            rel_delta = 0
            ABS = "0"
            REL = "0"
        
        # Eine saubere Zeile nur mit den Werten für LaTeX
        row = f"       {namM} & {VAL1} & {VAL2} & {ABS} & {REL} & {SIGMA} \\\\"
        table_rows.append(row)
    
    body_str = "\n".join(table_rows)
    
    # 3. Tabellenfuß
    output_footer = textwrap.dedent(f"""
                \\bottomrule
        \\end{{tabularx}}
        \\label{{table:Vergleich_{nam1}}} 
        \\end{{table}}
    """).strip()
    
    # Alles zusammensetzen
    final_output = f"{output_header}\n{body_str}\n{output_footer}"
    
    print(final_output)


In [8]:
# #Erstellung von Vergleichstabellen in Latex
# def compare_table_withliterature(nam1,nam2,einheiten,val1,err1,val2):
#     VAL1=round_to_sigs(val1,err1,r'')[2]
#     abs=np.abs(val1-val2)
#     abs_delta=np.sqrt(err1**2)
#     if abs_delta!=0:
#         SIGMA=sigma_abweichung(val1,err1,val2,0)[1]
#     else:
#         SIGMA=0.0
#     ABS=round_to_sigs(abs,abs_delta,r'')[2]
#     rel=abs/np.abs(val2)*100
#     rel_delta=rel*np.sqrt((abs_delta/abs)**2+(err1/val1)**2) if abs != 0 else 0
#     REL=round_to_sigs(rel,rel_delta,r'')[2]
#     output = textwrap.dedent(f"""
#         \\begin{{table}}[htb]
#         \\centering
#         \\caption{{}}
#         \\begin{{tabularx}}{{0.8\\textwidth}}{{ZZZZZ}}
#             \\toprule
#                 {nam1}[\\unit{{{einheiten}}}]&{nam2}[\\unit{{{einheiten}}}]&\\text{{abs.Abw.}}[\\unit{{{einheiten}}}]&\\text{{proz.Abw.}}[\\unit{{\\percent}}]&S[\\unitgma]\\\\\\midrule
#                 {VAL1}&\\num{{{val2}}}&{ABS}&{REL}&{SIGMA}\\\\
#             \\bottomrule
#         \\end{{tabularx}}
#         \\label{{table:Vergleich_{nam1}}} 
#         \\end{{table}}
#     """)
#     print(output)
    

    
# #Größenvergleich in latex
# def size_comp_str(name1,name2,val1,val2):
#     if val1<val2:
#         return name1+"<"+name2
#     elif val1>val2:
#         return name1+">"+name2
#     else:
#         return name1+"="+name2
   
#Erstellung von Vergleichstabellen in Latex
import textwrap 
def compare_table(nam1,nam2,einheiten,val1,err1,val2,err2):
    VAL1=round_to_sigs(val1,err1,r'')[2]
    if err2!=0:
        VAL2=round_to_sigs(val2,err2,r'')[2]
    else:
        VAL2=val2
    abs=np.abs(val1-val2)
    abs_delta=np.sqrt(err1**2+err2**2)
    if abs_delta!=0:
        SIGMA=sigma_abweichung(val1,err1,val2,err2)[1]
    else:
        SIGMA=0.0

    rel=abs/np.abs(val1)*100
    if abs != 0:
        rel_delta = rel * np.sqrt((abs_delta / abs)**2 + (err1 / val1)**2)
        ABS = round_to_sigs(abs, abs_delta, r'')[2]
        REL = round_to_sigs(rel, rel_delta, r'')[2]
    else:
        rel_delta = 0
        ABS = "0"
        REL = "0"

    output = textwrap.dedent(f"""
        \\begin{{table}}[htbp]
        \\centering
        \\caption{{Vergleich von \\({nam1}\\) und \\({nam2}\\)}}
        \\begin{{tabularx}}{{\\textwidth}}{{ZZZZx}}
            \\toprule
                {nam1}\\;[\\unit{{{einheiten}}}]&{nam2}\\;[\\unit{{{einheiten}}}]&\\text{{abs. Abw.}}\\;[\\unit{{{einheiten}}}]&\\text{{proz. Abw.}}\\;[\\unit{{\\percent}}]&S\\;[\\sigma]\\\\\\midrule
                {VAL1}&{VAL2}&{ABS}&{REL}&{SIGMA}\\\\
            \\bottomrule
        \\end{{tabularx}}
        \\label{{table:Vergleich_{nam1}}} 
        \\end{{table}}
    """)
    print(output)

In [9]:
compare_table(r'D_1',r'D_2',r'\Nm\per\radian',0.0218,0.0023,0.0225,0.0004)


\begin{table}[htbp]
\centering
\caption{Vergleich von \(D_1\) und \(D_2\)}
\begin{tabularx}{\textwidth}{ZZZZx}
    \toprule
        D_1\;[\unit{\Nm\per\radian}]&D_2\;[\unit{\Nm\per\radian}]&\text{abs. Abw.}\;[\unit{\Nm\per\radian}]&\text{proz. Abw.}\;[\unit{\percent}]&S\;[\sigma]\\\midrule
        \num{0.0218(0.0023)}&\num{0.0225(0.0004)}&\num{0.0007(0.0024)}&\num{3(11)}&\num{0.3}\\
    \bottomrule
\end{tabularx}
\label{table:Vergleich_D_1} 
\end{table}



## V14 Mathematisches Pendel

In [10]:
compare_table(r'g_{\text{exp}}',r'g_{\text{lit}}',r'\m\per\s\squared',9.810,0.014,9.80984,0.00002)
compare_table(r'g_{\text{korr}}',r'g_{\text{lit}}',r'\m\per\s\squared',9.822,0.014,9.80984,0.00002)


\begin{table}[htbp]
\centering
\caption{Vergleich von \(g_{\text{exp}}\) und \(g_{\text{lit}}\)}
\begin{tabularx}{\textwidth}{ZZZZx}
    \toprule
        g_{\text{exp}}\;[\unit{\m\per\s\squared}]&g_{\text{lit}}\;[\unit{\m\per\s\squared}]&\text{abs. Abw.}\;[\unit{\m\per\s\squared}]&\text{proz. Abw.}\;[\unit{\percent}]&S\;[\sigma]\\\midrule
        \num{9.810(0.014)}&\num{9.809840(0.000020)}&\num{0.000(0.015)}&\num{0.00(0.15)}&\num{0.012}\\
    \bottomrule
\end{tabularx}
\label{table:Vergleich_g_{\text{exp}}} 
\end{table}


\begin{table}[htbp]
\centering
\caption{Vergleich von \(g_{\text{korr}}\) und \(g_{\text{lit}}\)}
\begin{tabularx}{\textwidth}{ZZZZx}
    \toprule
        g_{\text{korr}}\;[\unit{\m\per\s\squared}]&g_{\text{lit}}\;[\unit{\m\per\s\squared}]&\text{abs. Abw.}\;[\unit{\m\per\s\squared}]&\text{proz. Abw.}\;[\unit{\percent}]&S\;[\sigma]\\\midrule
        \num{9.822(0.014)}&\num{9.809840(0.000020)}&\num{0.012(0.015)}&\num{0.12(0.15)}&\num{0.9}\\
    \bottomrule
\end{tabular

## V15 Schiefe Ebene

In [11]:
names=[r'a_H',r'a_V']
values_exp=np.array([0.918,0.750])
Delta_values_exp=np.array([0.010,0.008])
values_calc=np.array([0.968,0.777])
Delta_values_calc=np.array([0.023,0.018])
compare_table_array(names,r'a^{\text{exp}}',r'a^{\text{calc}}',r'\m\per\s\squared',values_exp,Delta_values_exp,values_calc,Delta_values_calc)

\begin{table}[htbp]
\centering
\caption{Vergleich von \(a^{\text{exp}}\) und \(a^{\text{calc}}\)}
\begin{tabularx}{\textwidth}{ZZZZZS[table-format=1.2]}
    \toprule
         \text{Messreihe} & a^{\text{exp}}\;[\unit{\m\per\s\squared}] & a^{\text{calc}}\;[\unit{\m\per\s\squared}] & \text{abs. Abw.}\;[\unit{\m\per\s\squared}] & \text{proz. Abw.}\;[\unit{\percent}] & {S\(\;[\sigma]\)} \\\midrule
       a_H & \num{0.918(0.010)} & \num{0.968(0.023)} & \num{0.05(0.03)} & \num{5(3)} & 2.0 \\
       a_V & \num{0.750(0.008)} & \num{0.777(0.018)} & \num{0.027(0.020)} & \num{4(3)} & 1.4 \\
\bottomrule
\end{tabularx}
\label{table:Vergleich_a^{\text{exp}}} 
\end{table}


## Versuch 21 Elektrolyse

In [12]:
names=[r'\text{Kathode}',r'\text{Anode}']
values_exp=np.array([9.4,9.1])
Delta_values_exp=np.array([0.4,0.4])
values_calc=np.array([9.648,9.648])
Delta_values_calc=np.array([0,0])
compare_table_array(names,r'F^{\text{exp}}',r'F^{\text{lit}}',r'\four\coulomb\per\mole',values_exp,Delta_values_exp,values_calc,Delta_values_calc)

\begin{table}[htbp]
\centering
\caption{Vergleich von \(F^{\text{exp}}\) und \(F^{\text{lit}}\)}
\begin{tabularx}{\textwidth}{ZZZZZS[table-format=1.2]}
    \toprule
         \text{Messreihe} & F^{\text{exp}}\;[\unit{\four\coulomb\per\mole}] & F^{\text{lit}}\;[\unit{\four\coulomb\per\mole}] & \text{abs. Abw.}\;[\unit{\four\coulomb\per\mole}] & \text{proz. Abw.}\;[\unit{\percent}] & {S\(\;[\sigma]\)} \\\midrule
       \text{Kathode} & \num{9.4(0.4)} & 9.648 & \num{0.2(0.4)} & \num{3(5)} & 0.7 \\
       \text{Anode} & \num{9.1(0.4)} & 9.648 & \num{0.5(0.4)} & \num{6(5)} & 1.4 \\
\bottomrule
\end{tabularx}
\label{table:Vergleich_F^{\text{exp}}} 
\end{table}


## Versuch 22 Millikan

In [13]:
compare_table(r'\bar{Q_1}',r'e',r'\mnineteen\coulomb',1.564,0.014,1.602176634,0)


\begin{table}[htbp]
\centering
\caption{Vergleich von \(\bar{Q_1}\) und \(e\)}
\begin{tabularx}{\textwidth}{ZZZZx}
    \toprule
        \bar{Q_1}\;[\unit{\mnineteen\coulomb}]&e\;[\unit{\mnineteen\coulomb}]&\text{abs. Abw.}\;[\unit{\mnineteen\coulomb}]&\text{proz. Abw.}\;[\unit{\percent}]&S\;[\sigma]\\\midrule
        \num{1.564(0.014)}&1.602176634&\num{0.038(0.014)}&\num{2.4(0.9)}&\num{3}\\
    \bottomrule
\end{tabularx}
\label{table:Vergleich_\bar{Q_1}} 
\end{table}



In [14]:
v_f=4.647*1e-5
v_s=2.203*1e-5
Delta_v_f=0.5*1e-5
Delta_v_s=0.2*1e-5
C_1=1.9987356090*1e-10
U=501
f=0.895
Delta_f=0.006
Q=C_1*np.sqrt(v_f*f**3)*(v_f + v_s)/U
Delta_Q=np.sqrt(C_1**2*f*(Delta_v_f**2*f**2*(3*v_f + v_s)**2 + v_f**2*(9*Delta_f**2*(v_f + v_s)**2 + 4*Delta_v_s**2*f**2))/(U**2*v_f))/2
Q,Delta_Q,_,latexQ=round_to_sigs(Q,Delta_Q,r'\coulomb')
print(latexQ)
gff("C_1*(v_f+v_s)*sqrt(v_f*f**3)/U","v_f,v_s,f")

\qty{1.58E-19(2.1E-20)}{\coulomb}
Funktion:


<IPython.core.display.Math object>

\frac{C_{1} \sqrt{f^{3} v_{f}} \left(v_{f} + v_{s}\right)}{U}
Absoluter Fehler:


<IPython.core.display.Math object>

\frac{\sqrt{\frac{C_{1}^{2} f \left(\Delta_{v f}^{2} f^{2} \left(3 v_{f} + v_{s}\right)^{2} + v_{f}^{2} \left(9 \Delta_{f}^{2} \left(v_{f} + v_{s}\right)^{2} + 4 \Delta_{v s}^{2} f^{2}\right)\right)}{U^{2} v_{f}}}}{2}
Relativer Fehler:


<IPython.core.display.Math object>

\frac{\sqrt{\frac{9 \Delta_{f}^{2}}{f^{2}} + \frac{\Delta_{v f}^{2} \left(3 v_{f} + v_{s}\right)^{2}}{v_{f}^{2} \left(v_{f} + v_{s}\right)^{2}} + \frac{4 \Delta_{v s}^{2}}{\left(v_{f} + v_{s}\right)^{2}}}}{2}


(C_1*sqrt(f**3*v_f)*(v_f + v_s)/U,
 sqrt(C_1**2*f*(Delta_v_f**2*f**2*(3*v_f + v_s)**2 + v_f**2*(9*Delta_f**2*(v_f + v_s)**2 + 4*Delta_v_s**2*f**2))/(U**2*v_f))/2,
 sqrt(9*Delta_f**2/f**2 + Delta_v_f**2*(3*v_f + v_s)**2/(v_f**2*(v_f + v_s)**2) + 4*Delta_v_s**2/(v_f + v_s)**2)/2,
 [(v_f, Delta_v_f), (v_s, Delta_v_s), (f, Delta_f)])

In [15]:
names=[r'\bar{Q}_{n_q=1}',r'\bar{Q}_{n=58}', r'\Delta \bar{Q}_{n_q=1}',r'\Delta \bar{Q}_{n_q=1}']
values_exp=np.array([1.56,1.58,1.564,1.585])
Delta_values_exp=np.array([0.06,0.07,0.014,0.009])
values_calc=np.full(4,1.602)
Delta_values_calc=np.full(4,0)
compare_table_array(names,r'\bar{Q}',r'e',r'\mnineteen\coulomb',values_exp,Delta_values_exp,values_calc,Delta_values_calc)

\begin{table}[htbp]
\centering
\caption{Vergleich von \(\bar{Q}\) und \(e\)}
\begin{tabularx}{\textwidth}{ZZZZZS[table-format=1.2]}
    \toprule
         \text{Messreihe} & \bar{Q}\;[\unit{\mnineteen\coulomb}] & e\;[\unit{\mnineteen\coulomb}] & \text{abs. Abw.}\;[\unit{\mnineteen\coulomb}] & \text{proz. Abw.}\;[\unit{\percent}] & {S\(\;[\sigma]\)} \\\midrule
       \bar{Q}_{n_q=1} & \num{1.56(0.06)} & 1.602 & \num{0.04(0.06)} & \num{3(4)} & 0.8 \\
       \bar{Q}_{n=58} & \num{1.58(0.07)} & 1.602 & \num{0.02(0.07)} & \num{1(5)} & 0.4 \\
       \Delta \bar{Q}_{n_q=1} & \num{1.564(0.014)} & 1.602 & \num{0.038(0.014)} & \num{2.4(0.9)} & 3.0 \\
       \Delta \bar{Q}_{n_q=1} & \num{1.585(0.009)} & 1.602 & \num{0.017(0.009)} & \num{1.1(0.6)} & 1.9 \\
\bottomrule
\end{tabularx}
\label{table:Vergleich_\bar{Q}} 
\end{table}


In [16]:
round_to_sigs(1100,80,r'')

(1100.0, 80.0, '\\num{1100(80)}', '\\num{1100(80)}')

## Versuch 26 - Schallgeschwindigkeit

In [17]:
names=[r'c^{\text{Luft}}',r'c^{\ce{CO_2}}', r'c_0^{\text{Luft}}',r'c_0^{\ce{CO_2}}']
values_exp=np.array([345.5,270.3,331.3,259.9])
Delta_values_exp=np.array([0.6,0.5,0.8,0.6])
values_calc=np.array([346,261,331,250])
Delta_values_calc=np.array([3,8,3,6])
compare_table_array(names,r'c^{\text{allg.}}',r'c^{\text{welle}}',r'\m\per\s',values_exp,Delta_values_exp,values_calc,Delta_values_calc)

\begin{table}[htbp]
\centering
\caption{Vergleich von \(c^{\text{allg.}}\) und \(c^{\text{welle}}\)}
\begin{tabularx}{\textwidth}{ZZZZZS[table-format=1.2]}
    \toprule
         \text{Messreihe} & c^{\text{allg.}}\;[\unit{\m\per\s}] & c^{\text{welle}}\;[\unit{\m\per\s}] & \text{abs. Abw.}\;[\unit{\m\per\s}] & \text{proz. Abw.}\;[\unit{\percent}] & {S\(\;[\sigma]\)} \\\midrule
       c^{\text{Luft}} & \num{345.5(0.6)} & \num{346(3)} & \num{1(4)} & \num{0.1(0.9)} & 0.17 \\
       c^{\ce{CO_2}} & \num{270.3(0.5)} & \num{261(8)} & \num{9(9)} & \num{3(3)} & 1.2 \\
       c_0^{\text{Luft}} & \num{331.3(0.8)} & \num{331(3)} & \num{0(4)} & \num{0.1(1.0)} & 0.1 \\
       c_0^{\ce{CO_2}} & \num{259.9(0.6)} & \num{250(6)} & \num{10(7)} & \num{3.8(2.4)} & 1.7 \\
\bottomrule
\end{tabularx}
\label{table:Vergleich_c^{\text{allg.}}} 
\end{table}


In [18]:
np.sqrt(1.40*44*1e-3/(1.3*29*1e-3))

np.float64(1.2782614187410608)

In [19]:
c_l=331.1
c_c=259.0
Delta_c_l=0.8
Delta_c_c=0.6
V=c_l/c_c
Delta_V=np.sqrt(Delta_c_c**2/c_c**2 + Delta_c_l**2/c_l**2)
V,Delta_V,_,latexV=round_to_sigs(V,Delta_V,r'')
print(latexV)
gff("c_l/c_c","c_l,c_c")

\num{1.278(0.004)}
Funktion:


<IPython.core.display.Math object>

\frac{c_{l}}{c_{c}}
Absoluter Fehler:


<IPython.core.display.Math object>

\sqrt{\frac{\Delta_{c c}^{2} c_{l}^{2} + \Delta_{c l}^{2} c_{c}^{2}}{c_{c}^{4}}}
Relativer Fehler:


<IPython.core.display.Math object>

\sqrt{\frac{\Delta_{c c}^{2}}{c_{c}^{2}} + \frac{\Delta_{c l}^{2}}{c_{l}^{2}}}


(c_l/c_c,
 sqrt((Delta_c_c**2*c_l**2 + Delta_c_l**2*c_c**2)/c_c**4),
 sqrt(Delta_c_c**2/c_c**2 + Delta_c_l**2/c_l**2),
 [(c_l, Delta_c_l), (c_c, Delta_c_c)])

In [20]:
compare_table(r'V^{\text{allg}}',r'V^{\text{welle}}',r'',1.32,0.03,1.278,0)


\begin{table}[htbp]
\centering
\caption{Vergleich von \(V^{\text{allg}}\) und \(V^{\text{welle}}\)}
\begin{tabularx}{\textwidth}{ZZZZx}
    \toprule
        V^{\text{allg}}\;[\unit{}]&V^{\text{welle}}\;[\unit{}]&\text{abs. Abw.}\;[\unit{}]&\text{proz. Abw.}\;[\unit{\percent}]&S\;[\sigma]\\\midrule
        \num{1.32(0.03)}&1.278&\num{0.04(0.03)}&\num{3.2(2.3)}&\num{1.5}\\
    \bottomrule
\end{tabularx}
\label{table:Vergleich_V^{\text{allg}}} 
\end{table}



## Versuch 33 Prismenspektrometer

In [21]:
names=np.array([1,2,3,4,5,6])
values_exp=np.array([442.0,470.5, 503.5,514.0,596.5,649.0])
Delta_values_exp=np.array([9.5,12.4,13.8,12.4,13.8,15.3])
values_calc=np.array([447.1,471.3,492.2,501.6,587.6,667.8 ])
Delta_values_calc=np.full(6,0)
compare_table_array(names,r'\lambda_{\text{exp}}',r'\lambda_{\text{lit}}',r'\nano\m',values_exp,Delta_values_exp,values_calc,Delta_values_calc)

\begin{table}[htbp]
\centering
\caption{Vergleich von \(\lambda_{\text{exp}}\) und \(\lambda_{\text{lit}}\)}
\begin{tabularx}{\textwidth}{ZZZZZS[table-format=1.2]}
    \toprule
         \text{Messreihe} & \lambda_{\text{exp}}\;[\unit{\nano\m}] & \lambda_{\text{lit}}\;[\unit{\nano\m}] & \text{abs. Abw.}\;[\unit{\nano\m}] & \text{proz. Abw.}\;[\unit{\percent}] & {S\(\;[\sigma]\)} \\\midrule
       1 & \num{442(10)} & 447.1 & \num{5(10)} & \num{1.2(2.2)} & 0.6 \\
       2 & \num{471(13)} & 471.3 & \num{1(13)} & \num{0(3)} & 0.07 \\
       3 & \num{504(14)} & 492.2 & \num{11(14)} & \num{2(3)} & 0.9 \\
       4 & \num{514(13)} & 501.6 & \num{12(13)} & \num{2.4(2.5)} & 1.0 \\
       5 & \num{597(14)} & 587.6 & \num{9(14)} & \num{1.5(2.4)} & 0.7 \\
       6 & \num{649(16)} & 667.8 & \num{19(16)} & \num{2.9(2.4)} & 1.3 \\
\bottomrule
\end{tabularx}
\label{table:Vergleich_\lambda_{\text{exp}}} 
\end{table}


In [22]:
names=np.array([1,2,3,4,5,6])
values_exp=np.array([634.0,492.5,429.5])
Delta_values_exp=np.array([18,16.5,10.5])
values_calc=np.array([656.3 ,486.1,434.0])
Delta_values_calc=np.full(3,0)
compare_table_array(names,r'\lambda_H^{\text{exp}}',r'\lambda_H^{\text{lit}}',r'\nano\m',values_exp,Delta_values_exp,values_calc,Delta_values_calc)

\begin{table}[htbp]
\centering
\caption{Vergleich von \(\lambda_H^{\text{exp}}\) und \(\lambda_H^{\text{lit}}\)}
\begin{tabularx}{\textwidth}{ZZZZZS[table-format=1.2]}
    \toprule
         \text{Messreihe} & \lambda_H^{\text{exp}}\;[\unit{\nano\m}] & \lambda_H^{\text{lit}}\;[\unit{\nano\m}] & \text{abs. Abw.}\;[\unit{\nano\m}] & \text{proz. Abw.}\;[\unit{\percent}] & {S\(\;[\sigma]\)} \\\midrule
       1 & \num{634(18)} & 656.3 & \num{22(18)} & \num{4(3)} & 1.3 \\
       2 & \num{493(17)} & 486.1 & \num{6(17)} & \num{1(4)} & 0.4 \\
       3 & \num{430(11)} & 434.0 & \num{5(11)} & \num{1.0(2.5)} & 0.5 \\
\bottomrule
\end{tabularx}
\label{table:Vergleich_\lambda_H^{\text{exp}}} 
\end{table}


In [23]:
names=[r'\bar{R}_\infty',r'\bar{R}_{\infty,\,\text{gm}}']
values_exp=np.array([1.110,1.114])
Delta_values_exp=np.array([0.017,0.018])
values_calc=np.array([1.09737,1.09737])
Delta_values_calc=np.full(2,0)
compare_table_array(names,r'R_\infty^{\text{exp}}',r'R^{\text{lit}}_\infty',r'\m',values_exp,Delta_values_exp,values_calc,Delta_values_calc)


\begin{table}[htbp]
\centering
\caption{Vergleich von \(R_\infty^{\text{exp}}\) und \(R^{\text{lit}}_\infty\)}
\begin{tabularx}{\textwidth}{ZZZZZS[table-format=1.2]}
    \toprule
         \text{Messreihe} & R_\infty^{\text{exp}}\;[\unit{\m}] & R^{\text{lit}}_\infty\;[\unit{\m}] & \text{abs. Abw.}\;[\unit{\m}] & \text{proz. Abw.}\;[\unit{\percent}] & {S\(\;[\sigma]\)} \\\midrule
       \bar{R}_\infty & \num{1.110(0.017)} & 1.09737 & \num{0.013(0.017)} & \num{1.1(1.6)} & 0.8 \\
       \bar{R}_{\infty,\,\text{gm}} & \num{1.114(0.018)} & 1.09737 & \num{0.017(0.018)} & \num{1.5(1.7)} & 1.0 \\
\bottomrule
\end{tabularx}
\label{table:Vergleich_R_\infty^{\text{exp}}} 
\end{table}


## Versuch 34 Spektralphotometrie

In [26]:
compare_table(r'\epsilon^{k^\prime}',r'\epsilon^{l}',r'\cm\squared\per\mole',2240,20,2680,110)


\begin{table}[htbp]
\centering
\caption{Vergleich von \(\epsilon^{k^\prime}\) und \(\epsilon^{l}\)}
\begin{tabularx}{\textwidth}{ZZZZx}
    \toprule
        \epsilon^{k^\prime}\;[\unit{\cm\squared\per\mole}]&\epsilon^{l}\;[\unit{\cm\squared\per\mole}]&\text{abs. Abw.}\;[\unit{\cm\squared\per\mole}]&\text{proz. Abw.}\;[\unit{\percent}]&S\;[\sigma]\\\midrule
        \num{2240(20)}&\num{2680(110)}&\num{440(120)}&\num{20(5)}&\num{4}\\
    \bottomrule
\end{tabularx}
\label{table:Vergleich_\epsilon^{k^\prime}} 
\end{table}



## Versuch 35 Fotoeffekt

In [28]:
names=[r'\text{LBC}',r'\text{Händisch}']
values_exp=np.array([6.2,6.6])
Delta_values_exp=np.array([1.5,0.8])
values_calc=np.array([6.6260689,6.6260689])
Delta_values_calc=np.full(2,0)
compare_table_array(names,r'h_\infty^{\text{exp}}',r'h^{\text{lit}}_\infty',r'\joule\s',values_exp,Delta_values_exp,values_calc,Delta_values_calc)


\begin{table}[htbp]
\centering
\caption{Vergleich von \(h_\infty^{\text{exp}}\) und \(h^{\text{lit}}_\infty\)}
\begin{tabularx}{\textwidth}{ZZZZZS[table-format=1.2]}
    \toprule
         \text{Messreihe} & h_\infty^{\text{exp}}\;[\unit{\joule\s}] & h^{\text{lit}}_\infty\;[\unit{\joule\s}] & \text{abs. Abw.}\;[\unit{\joule\s}] & \text{proz. Abw.}\;[\unit{\percent}] & {S\(\;[\sigma]\)} \\\midrule
       \text{LBC} & \num{6.2(1.5)} & 6.6260689 & \num{0.4(1.5)} & \num{7(25)} & 0.3 \\
       \text{Händisch} & \num{6.6(0.8)} & 6.6260689 & \num{0.0(0.8)} & \num{0(13)} & 0.04 \\
\bottomrule
\end{tabularx}
\label{table:Vergleich_h_\infty^{\text{exp}}} 
\end{table}


## Versuch 42 Wärmekapazität

In [29]:
sigma_abweichung(39,19,70,0)

(1.7, '\\num{1.7}')

In [30]:
compare_table(r'W_{\text{exp}}',r'W_{\text{lit}}',r'\J\per\K',39,19,70,0)



\begin{table}[htbp]
\centering
\caption{Vergleich von \(W_{\text{exp}}\) und \(W_{\text{lit}}\)}
\begin{tabularx}{\textwidth}{ZZZZx}
    \toprule
        W_{\text{exp}}\;[\unit{\J\per\K}]&W_{\text{lit}}\;[\unit{\J\per\K}]&\text{abs. Abw.}\;[\unit{\J\per\K}]&\text{proz. Abw.}\;[\unit{\percent}]&S\;[\sigma]\\\midrule
        \num{39(19)}&70&\num{31(19)}&\num{80(70)}&\num{1.7}\\
    \bottomrule
\end{tabularx}
\label{table:Vergleich_W_{\text{exp}}} 
\end{table}

